# CS-MON : Dataset audit and colour-preserving preprocessing

### Run manually in your existing Windows VS Code window

The Ubuntu GPU environment is installed. Connect this notebook through **Select Kernel ? Select Another Kernel ? Existing Jupyter Server**, enter the authenticated localhost URL supplied in chat, and select **CS-MON GPU (WSL)**. This uses Ubuntu without opening another VS Code window. The server must remain running; after a Windows/WSL restart it must be started again.

Run **01_preprocessing.ipynb ? Run All** and wait for completion. Then run **02_training_evaluation.ipynb ? Run All**. Training currently uses **quick mode** (one seed, CS-MON and four baselines); full multi-seed training and ablations require changing the configuration. Do not run both notebooks simultaneously. The GPU preflight runs automatically before training.

**Explainable Colour-Space VMamba-Ordinal Learning for Artificial Colour Intensity Estimation in Traditional Sweets**

Run this notebook first, from the project root. The real data are in `Dataset/`, joined by filename stem to `dataset_details.csv`; no images are moved or copied. Only `outputs/split_manifest.csv` is written. Dependencies: Python 3.10+, numpy, pandas, Pillow, matplotlib, seaborn, scikit-learn, scikit-image, tqdm, Jupyter.

**Scientific boundary:** intensity labels are supplied by the CSV, not inferred from pixels. The supplied file contains no controlled-preparation/reference-measurement provenance. Confirm and document that provenance before interpreting a research experiment. Device and lighting are available; physical sample, preparation batch and capture session are absent. Filename prefixes are not evidence of independent samples. The default split is explicitly exploratory and cannot establish sample-independent generalization. Whole images are used because there are no verified ROI masks. Image screening does not identify chemicals, measure dye concentration or establish food safety.

In [1]:
from pathlib import Path
import hashlib, json, warnings, itertools
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ExifTags
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.color import rgb2hsv, rgb2lab
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from IPython.display import display

CONFIG = dict(seed=42, image_size=224, data_dir='Dataset', metadata='dataset_details.csv',
              ratios=(0.70, 0.15, 0.15), near_hash_distance=4, split_trials=600,
              group_columns=['sample_id', 'preparation_batch_id', 'capture_session_id'],
              label_reference_documentation='', allow_exploratory_image_split=True)
ROOT = Path.cwd().resolve()
assert (ROOT / CONFIG['metadata']).is_file(), 'Run from the project root.'
LABELS = {'Low': 0, 'Medium': 1, 'High': 2}
SWEETS = ['Kesari', 'Muscat', 'Boondhi']
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams.update({'figure.dpi': 125, 'font.size': 10, 'axes.titleweight': 'bold',
                     'figure.constrained_layout.use': True})

## 1 : Join actual files to documented labels
No IDs or group metadata are synthesized. Audit component IDs below mean only connected duplicate/metadata relationships; they are not physical-sample IDs. Unsupported files, missing metadata, missing images and unreadable images are displayed and excluded from the usable set. Conflicting labels for duplicate images stop the pipeline.

In [ ]:
metadata = pd.read_csv(ROOT / CONFIG['metadata'], dtype=str).fillna('')
assert {'image_id','sweet','colouring_intensity'} <= set(metadata)
assert metadata.image_id.is_unique, 'Ambiguous duplicate metadata IDs.'
assert metadata.colouring_intensity.isin(LABELS).all(), 'Unknown intensity labels.'
assert metadata.sweet.isin(SWEETS).all(), 'Unknown sweet category.'
all_files = sorted(p for p in (ROOT / CONFIG['data_dir']).rglob('*') if p.is_file())
image_files = [p for p in all_files if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}]
files = pd.DataFrame({'image_id':[p.stem for p in image_files],
                      'relative_path':[p.relative_to(ROOT).as_posix() for p in image_files]})
assert files.image_id.is_unique, 'Filename stems must uniquely identify images.'
joined = metadata.merge(files, on='image_id', how='outer', indicator=True, validate='one_to_one')
display(joined.loc[joined['_merge'] != 'both'])
print('Unsupported files:', [p.name for p in all_files if p not in image_files])
print('Metadata columns:', list(metadata.columns))
missing_groups = [c for c in CONFIG['group_columns'] if c not in metadata or not metadata[c].str.strip().any()]
print('Unavailable dependence metadata:', missing_groups)
if not CONFIG['label_reference_documentation']:
    warnings.warn('Label provenance is unverified: no preparation/reference-measurement record was supplied.')

def load_rgb(path):
    with Image.open(ROOT / path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

def resize_rgb(im):
    # Whole-image square resize, common to every model; no photometric alteration.
    return im.resize((CONFIG['image_size'],)*2, Image.Resampling.LANCZOS)

def representations(im):
    rgb = np.asarray(resize_rgb(im), dtype=np.float32) / 255
    return rgb, rgb2hsv(rgb), rgb2lab(rgb)

def difference_hash(im):
    gray = np.asarray(im.convert('L').resize((9,8), Image.Resampling.LANCZOS))
    return int(''.join('1' if b else '0' for b in (gray[:,1:] > gray[:,:-1]).ravel()), 2)

records, corrupt = [], []
for row in tqdm(joined.loc[joined['_merge']=='both'].drop(columns='_merge').to_dict('records'), desc='Image audit'):
    try:
        path = ROOT / row['relative_path']
        with Image.open(path) as raw:
            mode, fmt = raw.mode, raw.format
            exif = {ExifTags.TAGS.get(k,str(k)):str(v) for k,v in raw.getexif().items() if k in (271,272)}
            raw.verify()
        im = load_rgb(row['relative_path'])
        rgb,hsv,lab = representations(im)
        gray = np.asarray(resize_rgb(im).convert('L'),dtype=np.float32)/255
        lap = -4*gray[1:-1,1:-1]+gray[:-2,1:-1]+gray[2:,1:-1]+gray[1:-1,:-2]+gray[1:-1,2:]
        records.append(dict(row, label=LABELS[row['colouring_intensity']], width=im.width,height=im.height,
            aspect_ratio=im.width/im.height, colour_mode=mode, file_format=fmt, exif_camera=json.dumps(exif),
            sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
            pixel_sha256=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest(), dhash=str(difference_hash(im)),
            brightness=float(gray.mean()), blur_laplacian_variance=float(lap.var()),
            underexposed=float((gray<0.02).mean()),overexposed=float((gray>0.98).mean()),
            mean_saturation=float(hsv[...,1].mean()),mean_value=float(hsv[...,2].mean()),
            mean_L=float(lab[...,0].mean()), mean_chroma=float(np.linalg.norm(lab[...,1:],axis=-1).mean())))
    except (OSError, ValueError, SyntaxError) as e:
        corrupt.append({'image_id':row['image_id'],'error':str(e)})
audit = pd.DataFrame(records).reset_index(drop=True)
assert len(audit), 'No readable, labelled images.'
display(pd.DataFrame(corrupt))
display(audit.groupby(['sweet','colouring_intensity']).size().unstack().reindex(index=SWEETS,columns=LABELS))
print(f'{len(image_files)} image files; {len(audit)} usable; {len(corrupt)} corrupt')

for column in ['device','lighting','colouring']:
    if column in metadata:
        display(metadata.groupby(column).size().rename('Image count').to_frame())

## 2 : Dataset overview, gallery and distributions
All displays use the actual usable images. Blur is an uncalibrated resized-image Laplacian variance, useful for relative quality inspection, not an exclusion rule.

In [ ]:
fig, axes = plt.subplots(1,4,figsize=(13,2.2))
values = [len(audit),audit.sweet.nunique(),audit.label.nunique(),
          audit['sample_id'].nunique() if 'sample_id' in audit else 'Unavailable']
for ax,title,value in zip(axes,['Usable images','Sweet categories','Intensity levels','Physical samples'],values):
    ax.axis('off'); ax.text(.5,.63,str(value),ha='center',fontsize=25,color='#176b87')
    ax.text(.5,.2,title,ha='center')
fig.suptitle('Dataset overview'); plt.show()
fig,axes=plt.subplots(3,3,figsize=(9,9))
for i,sweet in enumerate(SWEETS):
    for j,level in enumerate(LABELS):
        ax=axes[i,j]; subset=audit[(audit.sweet==sweet)&(audit.colouring_intensity==level)]
        if len(subset): ax.imshow(load_rgb(subset.iloc[0].relative_path))
        else: ax.text(.5,.5,'No images',ha='center')
        ax.set_title(f'{sweet} : {level}'); ax.axis('off')
fig.suptitle('Real dataset examples'); plt.show()
audit.groupby(['sweet','colouring_intensity']).size().unstack().reindex(index=SWEETS,columns=LABELS).plot.bar(figsize=(9,4),rot=0)
plt.title('Image counts by sweet and intensity'); plt.ylabel('Images'); plt.xlabel('Sweet'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
for ax,col in zip(axes,['width','height','aspect_ratio']):
    sns.histplot(data=audit,x=col,ax=ax); ax.set_title(col.replace('_',' ').title())
fig.suptitle('Image resolution analysis'); plt.show()

## 3 : Colour-space exploration and preprocessing
RGB is sRGB in [0,1]. HSV uses hue in [0,1] (one complete turn), saturation/value in [0,1]. CIELAB uses the D65 white point, L* in [0,100], a*/b* approximately [-128,127]. Training maps RGB and HSV to [-1,1], and LAB to `(L*/50-1, a*/128, b*/128)` for the projections. Fused RGB receives ImageNet mean/std normalization exactly once, immediately before the pretrained backbone. These are three representations of each observation.

Whole-image resizing to 224 x 224 is explicit; no ROI is inferred. No histogram equalization, white balance, hue jitter or brightness adjustment is applied. LAB and HSV channels are shown as scalar fields with scales, never interpreted as RGB arrays.

In [ ]:
example = load_rgb(audit.iloc[0].relative_path)
rgb,hsv,lab = representations(example)
fig,axes=plt.subplots(3,4,figsize=(13,8))
for row,(space,array,names,ranges) in enumerate([
    ('RGB',rgb,['R','G','B'],[(0,1)]*3),('HSV',hsv,['Hue','Saturation','Value'],[(0,1)]*3),
    ('CIELAB',lab,['L*','a*','b*'],[(0,100),(-128,127),(-128,127)])]):
    axes[row,0].imshow(rgb); axes[row,0].set_title(f'Source RGB : {space}'); axes[row,0].axis('off')
    for j in range(3):
        ax=axes[row,j+1]; artist=ax.imshow(array[...,j],cmap='twilight' if space=='HSV' and j==0 else 'viridis',vmin=ranges[j][0],vmax=ranges[j][1])
        ax.set_title(names[j]); ax.axis('off'); fig.colorbar(artist,ax=ax,shrink=.7)
fig.suptitle('Colour channels with explicit numeric scales'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
for ax,array,names,title in zip(axes,[rgb,hsv,lab],[['R','G','B'],['H','S','V'],['L*','a*','b*']],['RGB','HSV','CIELAB']):
    for j,name in enumerate(names): ax.hist(array[...,j].ravel(),bins=50,density=True,histtype='step',label=name)
    ax.set_title(title+' : example image'); ax.set_xlabel('Channel value'); ax.set_ylabel('Density'); ax.legend()
plt.show()
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axes,['mean_saturation','mean_L','mean_chroma']):
    sns.boxplot(data=audit,x='colouring_intensity',y=col,hue='sweet',order=list(LABELS),ax=ax,fliersize=1)
    ax.set_xlabel('Supplied intensity'); ax.set_title(col.replace('_',' ').title())
plt.show()
fig,axes=plt.subplots(1,4,figsize=(13,4))
for ax,im,title in zip(axes,[example,example,resize_rgb(example),rgb],['Oriented RGB','ROI: whole image','Resized','Final RGB [0,1]']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.show()
fig,axes=plt.subplots(1,4,figsize=(15,3.5))
for ax,col in zip(axes,['brightness','blur_laplacian_variance','underexposed','overexposed']):
    sns.histplot(data=audit,x=col,hue='colouring_intensity',ax=ax,element='step'); ax.set_title(col.replace('_',' ').title())
plt.show()

## 4 : Duplicate components and dependence constraints
Byte hashes and decoded, EXIF-corrected pixel hashes detect exact duplicates. A 64-bit difference hash produces **near-duplicate candidates** (Hamming distance <= configured threshold). These candidates are conservatively kept together without claiming that the hash identifies physical samples. False positive connections can reduce split flexibility. Images are not deleted. Metadata groups, if later supplied, are unioned transitively across every available dependence column; duplicate edges can join otherwise distinct metadata groups. Missing metadata never creates a single artificial group.

In [ ]:
parent=list(range(len(audit)))
def find(i):
    while parent[i]!=i:
        parent[i]=parent[parent[i]]; i=parent[i]
    return i
def union(i,j):
    a,b=find(i),find(j)
    if a!=b: parent[b]=a
exact_pairs=[]
for col in ['sha256','pixel_sha256']:
    for indices in audit.groupby(col).indices.values():
        for j in indices[1:]:
            i=int(indices[0]); j=int(j)
            assert audit.loc[i,'label']==audit.loc[j,'label'], 'Exact duplicate has conflicting intensity labels.'
            union(i,j); exact_pairs.append((i,j))
near_pairs=[]
hashes=[int(x) for x in audit.dhash]
for i in range(len(audit)):
    for j in range(i):
        if (hashes[i]^hashes[j]).bit_count() <= CONFIG['near_hash_distance']:
            near_pairs.append((j,i)); union(i,j)
reliable_columns=[]
for col in CONFIG['group_columns']:
    if col in audit and audit[col].fillna('').str.strip().ne('').any():
        reliable_columns.append(col)
        valid=audit[col].fillna('').str.strip().ne('')
        for indices in audit.loc[valid].groupby(col).groups.values():
            indices=list(indices)
            for j in indices[1:]: union(indices[0],j)
audit['audit_component']=[f'component_{find(i):06d}' for i in range(len(audit))]
fully_grouped=bool(reliable_columns) and audit[reliable_columns].fillna('').apply(lambda s:s.str.strip().ne('')).any(axis=1).all()
split_scope='metadata_constrained' if fully_grouped else 'exploratory_duplicate_constrained'
print('Exact duplicate links:',len(set(exact_pairs)), '| Near candidates:',len(near_pairs), '| Components:',audit.audit_component.nunique())
print('Split scope:',split_scope)
if not fully_grouped:
    warnings.warn('Missing or incomplete physical dependence metadata. This split cannot prove sample-independent generalization.')
    assert CONFIG['allow_exploratory_image_split'], 'Supply reliable grouping metadata or explicitly allow exploratory splitting.'
if near_pairs:
    fig,axes=plt.subplots(min(3,len(near_pairs)),2,figsize=(7,3*min(3,len(near_pairs))),squeeze=False)
    for row,(i,j) in enumerate(near_pairs[:3]):
        for ax,k in zip(axes[row],[i,j]):
            ax.imshow(load_rgb(audit.loc[k,'relative_path'])); ax.set_title(audit.loc[k,'image_id']); ax.axis('off')
    fig.suptitle('Near-duplicate candidates : inspect visually'); plt.show()

## 5 : Reproducible approximately stratified 70/15/15 split
Search random group assignments using only label/category counts and component sizes. Minimize deviation from target sweet x intensity counts; require every cell in every split. No image content or model performance selects the split. Exact ratios may be impossible with dependence constraints. A clear error is raised when adequate coverage cannot be obtained. The test set remains held out for the predeclared final comparisons.

In [ ]:
ratios=np.asarray(CONFIG['ratios'],dtype=float)
assert np.isclose(ratios.sum(),1) and (ratios>0).all()
strata=(audit.sweet+'|'+audit.colouring_intensity)
levels=sorted(strata.unique()); total=strata.value_counts().reindex(levels).to_numpy()
best=None; best_score=float('inf')
for trial in range(CONFIG['split_trials']):
    try:
        first=GroupShuffleSplit(n_splits=1,train_size=float(ratios[0]),random_state=CONFIG['seed']+trial)
        tr,rest=next(first.split(audit,groups=audit.audit_component))
        second=GroupShuffleSplit(n_splits=1,train_size=float(ratios[1]/ratios[1:].sum()),random_state=CONFIG['seed']+10000+trial)
        va0,te0=next(second.split(audit.iloc[rest],groups=audit.iloc[rest].audit_component))
        pieces=[tr,rest[va0],rest[te0]]
        counts=np.stack([strata.iloc[idx].value_counts().reindex(levels,fill_value=0).to_numpy() for idx in pieces])
        if (counts==0).any(): continue
        score=float(np.mean(((counts-ratios[:,None]*total)/(ratios[:,None]*total))**2))
        if score<best_score: best_score,best=score,pieces
    except ValueError:
        continue
assert best is not None, 'Cannot make three group-disjoint splits with all sweet/intensity cells. Supply more independent groups or review hash candidates.'
audit['split']=''
for split,idx in zip(['train','validation','test'],best): audit.loc[idx,'split']=split
for col in ['audit_component','sha256','pixel_sha256']+reliable_columns:
    valid=audit[col].fillna('').astype(str).str.strip().ne('')
    assert audit.loc[valid].groupby(col).split.nunique().max()==1, f'Leakage in {col}'
assert all(audit.loc[i,'split']==audit.loc[j,'split'] for i,j in near_pairs)
audit['split_scope']=split_scope
audit['split_seed']=CONFIG['seed']
audit['label_provenance']=CONFIG['label_reference_documentation'] or 'UNVERIFIED: supplied CSV labels only'
manifest_columns=['relative_path','image_id','sweet','colouring_intensity','label','split','audit_component',
                  'sha256','pixel_sha256','dhash','split_scope','split_seed','label_provenance']
manifest_columns += [c for c in metadata.columns if c not in manifest_columns]
manifest=audit[manifest_columns].copy()
output=ROOT/'outputs'; output.mkdir(exist_ok=True)
manifest.to_csv(output/'split_manifest.csv',index=False)
display(pd.crosstab([audit.sweet,audit.colouring_intensity],audit.split))
print('Actual proportions:',audit.split.value_counts(normalize=True).round(4).to_dict())
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,split in zip(axes,['train','validation','test']):
    counts=pd.crosstab(audit.loc[audit.split==split,'sweet'],audit.loc[audit.split==split,'colouring_intensity']).reindex(index=SWEETS,columns=LABELS,fill_value=0)
    counts.plot.bar(ax=ax,rot=0); ax.set_title(split.title()); ax.set_ylabel('Images')
plt.show()
fig,ax=plt.subplots(figsize=(9,3.5))
ax.bar(['Exact duplicate links','Near candidates','Cross-split duplicate links','Group overlap'],[len(set(exact_pairs)),len(near_pairs),0,0])
ax.set_ylabel('Count'); ax.set_title('Duplicate and leakage audit : detected relationships only'); plt.show()
print('Saved:',output/'split_manifest.csv')
print('Missing sample/batch/session metadata and unverified reference labels remain research limitations.')

## Verified dataset audit

All 1,944 supplied images were readable and labelled. The audit found no exact byte/pixel duplicates and 104 near-duplicate candidate pairs, producing 1,842 conservative audit components. The reproducible manifest contains **1356 train / 292 validation / 296 test** images. Every sweet/intensity combination appears in every split; all detected duplicate components and available dependence constraints are split-disjoint.

The user confirmed that physical sample/batch/session IDs and label-reference documentation are currently unavailable. These remain explicit limitations: the manifest is exploratory and does not establish sample-independent generalization or verified chemical ground truth. Original images and metadata were retained unchanged. Figures are embedded inline; only the manifest is generated by this notebook.


## 6 : Additional descriptive visualisations
These supplementary views use the existing `audit` table after the split is complete. They do not modify images, labels, preprocessing, duplicate components, or split assignments. Quality summaries describe whole resized images, including their backgrounds; they are not exclusion thresholds or estimates of dye concentration. Split diagnostics are descriptive and do not select or tune the split.

### Class balance within each sweet
Counts show dataset coverage; row percentages show the supplied intensity mix within each sweet.

In [ ]:
visual_counts = pd.crosstab(audit['sweet'], audit['colouring_intensity']).reindex(
    index=SWEETS, columns=list(LABELS), fill_value=0)
visual_percent = visual_counts.div(visual_counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(visual_counts, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            cbar_kws={'label': 'Images'})
sns.heatmap(visual_percent, annot=True, fmt='.1f', cmap='YlGnBu', vmin=0, vmax=100,
            ax=axes[1], cbar_kws={'label': 'Percent within sweet'})
for ax, title in zip(axes, ['Sweet × intensity: image counts', 'Intensity composition within each sweet (%)']):
    ax.set(title=title, xlabel='Supplied intensity', ylabel='Sweet')
plt.show()

### Resolution and quality relationships
Each scatter point represents one usable image. The diagonal marks square images before resizing; it does not prescribe a crop. Spearman correlations summarize monotonic relationships between existing audit measurements. Constant measurements have undefined correlations and appear blank. No observations are removed based on these plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=audit, x='width', y='height', hue='sweet', hue_order=SWEETS,
                alpha=.45, s=22, ax=axes[0])
visual_limits = [float(audit[['width', 'height']].min().min()),
                 float(audit[['width', 'height']].max().max())]
axes[0].plot(visual_limits, visual_limits, color='gray', linestyle=':', label='Square image')
axes[0].set(title='Original oriented image dimensions', xlabel='Width (pixels)', ylabel='Height (pixels)')
axes[0].legend()
sns.scatterplot(data=audit, x='brightness', y='blur_laplacian_variance',
                hue='colouring_intensity', hue_order=list(LABELS), style='sweet',
                style_order=SWEETS, alpha=.45, s=24, ax=axes[1])
axes[1].set_yscale('symlog', linthresh=1e-6)
axes[1].set(title='Brightness and relative sharpness after resizing',
            xlabel='Mean grayscale brightness [0, 1]', ylabel='Laplacian variance (symlog)')
plt.show()

visual_quality_columns = ['aspect_ratio', 'brightness', 'blur_laplacian_variance',
                          'underexposed', 'overexposed', 'mean_saturation', 'mean_L', 'mean_chroma']
visual_quality_labels = ['Aspect ratio', 'Brightness', 'Laplacian variance',
                         'Dark-pixel fraction', 'Bright-pixel fraction', 'Mean saturation', 'Mean L*', 'Mean chroma']
visual_correlation = audit[visual_quality_columns].corr(method='spearman')
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(visual_correlation, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, center=0,
            mask=visual_correlation.isna(), xticklabels=visual_quality_labels,
            yticklabels=visual_quality_labels, ax=ax, cbar_kws={'label': 'Spearman correlation'})
ax.set_title('Relationships between existing image audit measurements')
ax.tick_params(axis='x', rotation=40)
plt.show()

### Split composition and audit-component sizes
The allocation heatmap shows what percentage of each sweet/intensity stratum falls into each split, with the configured target in each column label. The component-size histogram counts connected audit components, which are not verified physical samples. These views retain the existing split and grouping approach.

In [ ]:
visual_split_order = ['train', 'validation', 'test']
visual_stratum_index = pd.MultiIndex.from_product(
    [SWEETS, list(LABELS)], names=['sweet', 'colouring_intensity'])
visual_split_counts = pd.crosstab([audit['sweet'], audit['colouring_intensity']], audit['split']).reindex(
    index=visual_stratum_index, columns=visual_split_order, fill_value=0)
visual_split_percent = visual_split_counts.div(
    visual_split_counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(visual_split_percent, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100,
            yticklabels=[f'{sweet} / {level}' for sweet, level in visual_stratum_index],
            xticklabels=[f'{name.title()}\n(target {target:.0%})'
                         for name, target in zip(visual_split_order, CONFIG['ratios'])],
            ax=axes[0], cbar_kws={'label': 'Percent of stratum'})
axes[0].set(title='Actual split allocation within each stratum (%)', xlabel='Split', ylabel='Sweet / supplied intensity')
visual_component_sizes = audit.groupby('audit_component').size()
visual_size_counts = visual_component_sizes.value_counts().sort_index()
axes[1].bar(visual_size_counts.index.astype(str), visual_size_counts.to_numpy(), color='#176b87')
for visual_bar, visual_count in enumerate(visual_size_counts.to_numpy()):
    axes[1].annotate(str(visual_count), (visual_bar, visual_count), xytext=(0, 4),
                     textcoords='offset points', ha='center', fontsize=9)
axes[1].margins(y=.15)
axes[1].set(title='Sizes of connected audit components', xlabel='Images per component', ylabel='Number of components')
plt.show()